# Mass Spectrum and Hessian Analysis

**What's in this notebook?** This notebook demonstrates how to compute the scalar mass spectrum at flux vacua using the Hessian of the scalar potential. It covers the four Hessian computation modes available in JAXVacua, the mass matrix after canonical normalisation, eigenvalue analysis, and mass hierarchies.

**In this notebook, you will learn:**
- How to compute the Hessian of the scalar potential via `model.hessian(mode=...)`
- The four computation modes: `None` (autodiff), `"SUSY"`, `"SUGRA"`, `"real"`
- How to obtain the canonically normalised mass matrix via `model.mass_matrix`
- How to analyse eigenvalue spectra and identify light/heavy directions
- How to verify consistency between Hessian implementations

**Prerequisites:** [Tutorial 5: Finding flux vacua](../02_vacuum_finding/5_finding_flux_vacua.ipynb).

**Related:** [hessian_SUGRA_analysis](../../notebooks/hessian_SUGRA_analysis.ipynb) — detailed derivation and debugging of the SUGRA Hessian formula.

(*Created:* March 2026)

## Imports and setup

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)
from jax import vmap
import time

import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams.update({"figure.dpi": 200, "font.size": 10, "axes.labelsize": 11,
                      "figure.figsize": (7, 3), "legend.frameon": False,
                      "xtick.direction": "in", "ytick.direction": "in"})

import jaxvacua as jvc

## Model and test points

We set up a two-modulus LCS model and prepare two types of test points:
1. **A generic point** — random moduli and fluxes where $D_AW \neq 0$
2. **A SUSY vacuum** — a true solution to $D_AW = 0$ found by Newton's method

In [ ]:
# Model
model = jvc.FluxVacuaFinder(h12=2, model_ID=1, maximum_degree=2, limit="LCS", model_type="KS")

# ── Generic point ──
rng = np.random.default_rng(42)
z_gen  = jnp.array(rng.uniform(-0.3, 0.3, 2) + 1j * rng.uniform(2, 4, 2))
zc_gen = jnp.conj(z_gen)
tau_gen  = complex(rng.uniform(-0.3, 0.3), rng.uniform(2, 5))
tauc_gen = jnp.conj(tau_gen)
fl_gen = jnp.array(rng.integers(-5, 6, 2 * model.n_fluxes).astype(float))

DW_gen = model.DW(z_gen, zc_gen, tau_gen, tauc_gen, fl_gen)
print(f"Generic point: |DW| = {float(jnp.sum(jnp.abs(DW_gen))):.2f}")

# ── SUSY vacuum ──
sampler = jvc.data_sampler(model, flux_bounds=[-8, 8], moduli_bounds=[1, 4], dilaton_bounds=[1, 4])
rns_key = jvc.PRNGSequence(123)
moduli_all, tau_all, fluxes_all, res_all = model.sample_SUSY_flux_vacua(
    N=20, sampler=sampler, rns_key=rns_key,
    max_iters=100, tol=1e-12, max_tadpole=model.D3_tadpole,
    mode="ISD", vmap_dim=20, print_progress=False,
)

# Pick the vacuum with smallest residual
best = int(jnp.argmin(res_all))
z_susy  = moduli_all[best]
zc_susy = jnp.conj(z_susy)
tau_susy  = tau_all[best]
tauc_susy = jnp.conj(tau_susy)
fl_susy = fluxes_all[best]

DW_susy = model.DW(z_susy, zc_susy, tau_susy, tauc_susy, fl_susy)
W0 = model.superpotential_gauge_invariant(z_susy, tau_susy, fl_susy)
print(f"SUSY vacuum:   |DW| = {float(jnp.sum(jnp.abs(DW_susy))):.2e},  |W0| = {float(jnp.abs(W0)):.4f}")
print(f"  z  = {z_susy}")
print(f"  τ  = {tau_susy}")

## The Hessian: four computation modes

The `FluxEFT` class provides four modes for computing the Hessian via `model.hessian(mode=...)`:

| Mode | Method | Valid at | Approach |
|------|--------|----------|----------|
| `None` | `_hessian_general` | Any point | `jacrev(dV)` — exact autodiff |
| `"SUSY"` | `_hessian_SUSY` | $D_AW = 0$ only | Analytical SUGRA formula (simplified at SUSY) |
| `"SUGRA"` | `_hessian_SUGRA` | Any point | Explicit SUGRA building blocks ($\Gamma$, $R$, $D_IW$, $K$, $W$) |
| `"real"` | `ddV_x` | Any point | Hessian in real coordinates $(\mathrm{Re}\,z, \mathrm{Im}\,z, \ldots)$ |

All modes should agree at SUSY points. At generic points, `"SUSY"` mode will differ.

### Comparison at the generic point

In [ ]:
# Adapted from hessian_SUGRA_analysis.ipynb, Section 12

modes = [None, "SUSY", "SUGRA", "real"]

print("Generic point (|DW| >> 0):")
print(f"|DW| = {float(jnp.sum(jnp.abs(DW_gen))):.2f}\n")
print(f"{'Mode':<10} {'Time (ms)':>10} {'Max |err| vs None':>20} {'Rel err':>12} {'Eigenvalues (real part)':>30}")
print("-" * 90)

H_ref = None
for mode in modes:
    # Warm up JIT
    _ = model.hessian(z_gen, zc_gen, tau_gen, tauc_gen, fl_gen, noscale=True, mode=mode)
    
    t0 = time.perf_counter()
    H = model.hessian(z_gen, zc_gen, tau_gen, tauc_gen, fl_gen, noscale=True, mode=mode)
    jax.block_until_ready(H)
    dt = (time.perf_counter() - t0) * 1000
    
    eigs = jnp.sort(jnp.linalg.eigvals(H).real)
    
    if mode == "real":
        eigs = eigs/2
    if mode is None:
        H_ref = H
        err_str = "—"
        rel_str = "—"
    else:
        if mode == "real":
            # Real mode has different shape — compare eigenvalues instead
            eigs_ref = jnp.sort(jnp.linalg.eigvals(H_ref).real)
            err = float(jnp.max(jnp.abs(eigs_ref - eigs)))
            scale = float(jnp.max(jnp.abs(eigs_ref)))
        else:
            err = float(jnp.max(jnp.abs(H_ref - H)))
            scale = float(jnp.max(jnp.abs(H_ref)))
        err_str = f"{err:.2e}"
        rel_str = f"{err/scale:.2e}" if scale > 0 else "—"
    
    eig_str = ", ".join(f"{float(e):.2f}" for e in eigs[:4])
    label = str(mode) if mode else "None (autodiff)"
    print(f"{label:<16} {dt:8.1f}ms   {err_str:>18}   {rel_str:>10}   [{eig_str}]")

```{note}
The `"SUGRA"` mode shows a relative error of $\sim 10^{-8}$ against the autodiff reference (`mode=None`). This is **not a bug** — it reflects the accumulated floating-point error from chaining many intermediate autodiff operations: the SUGRA formula builds the Hessian from third derivatives of the Kähler potential ($\partial_j\partial_k\partial_{\bar l} K$), Christoffel symbols, the Riemann tensor, and a 9-term product-rule expansion. Each differentiation step loses $\sim 1$–$2$ digits of precision.

In contrast, `mode=None` computes `jacrev(dV)` in a single second-order autodiff pass, reaching machine precision ($\sim 10^{-15}$).

For practical purposes, $10^{-8}$ relative error is negligible — it is well below the sensitivity of any physical observable. Use `mode="SUGRA"` when you need access to individual SUGRA building blocks ($D_IW$, $\Gamma^i_{jk}$, $R_{i\bar j k\bar l}$); use `mode=None` when maximum numerical precision is required.
```

### Comparison at the SUSY vacuum

In [ ]:
print("SUSY vacuum (|DW| ≈ 0):")
print(f"|DW| = {float(jnp.sum(jnp.abs(DW_susy))):.2e}\n")
print(f"{'Mode':<10} {'Time (ms)':>10} {'Max |err| vs None':>20} {'Rel err':>12}")
print("-" * 65)

H_ref_susy = None
for mode in modes:
    _ = model.hessian(z_susy, zc_susy, tau_susy, tauc_susy, fl_susy, noscale=True, mode=mode)
    
    t0 = time.perf_counter()
    H = model.hessian(z_susy, zc_susy, tau_susy, tauc_susy, fl_susy, noscale=True, mode=mode)
    jax.block_until_ready(H)
    dt = (time.perf_counter() - t0) * 1000
    
    if mode is None:
        H_ref_susy = H
        print(f"{'None (autodiff)':<16} {dt:8.1f}ms   {'—':>18}   {'—':>10}")
    else:
        if mode == "real":
            eigs_ref = jnp.sort(jnp.linalg.eigvals(H_ref_susy).real)
            eigs = jnp.sort(jnp.linalg.eigvals(H).real)/2
            err = float(jnp.max(jnp.abs(eigs_ref - eigs)))
            scale = float(jnp.max(jnp.abs(eigs_ref)))
        else:
            err = float(jnp.max(jnp.abs(H_ref_susy - H)))
            scale = float(jnp.max(jnp.abs(H_ref_susy)))
        rel = err/scale if scale > 0 else 0
        print(f"{str(mode):<16} {dt:8.1f}ms   {err:>18.2e}   {rel:>10.2e}")

print("\n→ All modes agree at SUSY (as expected).")

## Stress test: random points

Verify `hessian(mode="SUGRA")` matches `hessian(mode=None)` across 50 random field-space points.
This is adapted from [hessian_SUGRA_analysis](../../notebooks/hessian_SUGRA_analysis.ipynb), Section 11.

In [ ]:
rng = np.random.default_rng(42)
n_tests = 50
rel_errors = []

for i in range(n_tests):
    re_z = rng.uniform(-0.5, 0.5, model.h12)
    im_z = rng.uniform(2.0, 5.0, model.h12)
    z_i = jnp.array(re_z + 1j * im_z)
    zc_i = jnp.conj(z_i)
    tau_i = complex(rng.uniform(-0.4, 0.4), rng.uniform(2, 6))
    tauc_i = jnp.conj(tau_i)
    fl_i = jnp.array(rng.integers(-8, 9, 2 * model.n_fluxes).astype(float))
    ns = bool(rng.choice([True, False]))
    
    H_ref = model._hessian_general(z_i, zc_i, tau_i, tauc_i, fl_i, noscale=ns)
    H_sug = model._hessian_SUGRA(z_i, zc_i, tau_i, tauc_i, fl_i, noscale=ns)
    
    err = float(jnp.max(jnp.abs(H_ref - H_sug)))
    scale = float(jnp.max(jnp.abs(H_ref)))
    rel_errors.append(err / scale if scale > 0 else 0)

rel_errors = np.array(rel_errors)
print(f"Tested {n_tests} random points:")
print(f"  Max relative error: {rel_errors.max():.2e}")
print(f"  Mean relative error: {rel_errors.mean():.2e}")
print(f"  All < 1e-10: {np.all(rel_errors < 1e-10)} ✓")

## Mass matrix and eigenvalue spectrum

The **mass matrix** is the Hessian after canonical normalisation by the Kähler metric:

$$
M^2_{A\bar B} = K^{A\bar C}\, V_{C\bar B}
$$

The eigenvalues of $M^2$ give the physical scalar masses-squared. At a SUSY Minkowski vacuum ($V=0$, $DW=0$), we expect:
- All eigenvalues $\geq 0$ (positive semi-definite)
- A mass hierarchy controlled by the flux choice

In [ ]:
# Mass matrix at the SUSY vacuum
M2 = model.mass_matrix(z_susy, zc_susy, tau_susy, tauc_susy, fl_susy, mode=None, noscale=True)
eigenvalues = jnp.sort(jnp.linalg.eigvals(M2).real)

print("Mass matrix eigenvalues at SUSY vacuum:")
for i, ev in enumerate(eigenvalues):
    print(f"  m²_{i} = {float(ev):+.4e}")

print(f"\nAll ≥ 0: {bool(jnp.all(eigenvalues > -1e-8))} ✓")
print(f"Hierarchy: m²_max / m²_min = {float(eigenvalues[-1] / eigenvalues[0]):.1f}")

## Eigenvalue distributions across an ensemble

Compute the mass spectrum at every vacuum in our sample and visualise the distributions.

In [ ]:
# Compute mass eigenvalues for all vacua
all_eigenvalues = []
all_W0 = []

for i in range(len(moduli_all)):
    z_i = moduli_all[i]
    tau_i = tau_all[i]
    fl_i = fluxes_all[i]
    
    M2_i = model.mass_matrix(z_i, jnp.conj(z_i), tau_i, jnp.conj(tau_i), fl_i,
                              mode="SUSY", noscale=True)
    eigs_i = jnp.sort(jnp.linalg.eigvals(M2_i).real)
    all_eigenvalues.append(np.array(eigs_i))
    all_W0.append(float(jnp.abs(model.superpotential_gauge_invariant(z_i, tau_i, fl_i))))

all_eigenvalues = np.array(all_eigenvalues)
all_W0 = np.array(all_W0)

n_fields = all_eigenvalues.shape[1]
print(f"Computed mass spectra for {len(all_eigenvalues)} vacua ({n_fields} fields each)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7, 3))

colors = plt.cm.viridis(np.linspace(0.2, 0.9, n_fields))

# (a) Histogram of eigenvalues by field index
for j in range(n_fields):
    ev_j = all_eigenvalues[:, j]
    ev_j = ev_j[np.isfinite(ev_j) & (ev_j > 0)]
    if len(ev_j) > 0:
        axes[0].hist(np.log10(ev_j), bins=25, alpha=0.5, color=colors[j],
                     label=f"$m^2_{j}$", edgecolor="white", linewidth=0.3)

axes[0].set_xlabel(r"$\log_{10}(m^2)$")
axes[0].set_ylabel("Count")
axes[0].legend(fontsize=8, ncol=2)
axes[0].text(0.05, 0.92, "(a)", transform=axes[0].transAxes, fontsize=12, fontweight="bold")

# (b) Mass hierarchy: lightest vs heaviest
m2_min = np.min(all_eigenvalues, axis=1)
m2_max = np.max(all_eigenvalues, axis=1)
valid = (m2_min > 0) & np.isfinite(m2_min) & np.isfinite(m2_max)

axes[1].scatter(np.log10(m2_min[valid]), np.log10(m2_max[valid]),
                s=10, alpha=0.6, c=all_W0[valid], cmap="viridis",
                edgecolors="none", rasterized=True)
axes[1].plot([-5, 10], [-5, 10], 'k--', lw=0.5, alpha=0.3, label="$m^2_{\mathrm{min}} = m^2_{\mathrm{max}}$")
axes[1].set_xlabel(r"$\log_{10}(m^2_{\mathrm{min}})$")
axes[1].set_ylabel(r"$\log_{10}(m^2_{\mathrm{max}})$")
cb = fig.colorbar(axes[1].collections[0], ax=axes[1], shrink=0.9, pad=0.02)
cb.set_label(r"$|W_0|$", fontsize=10)
cb.ax.tick_params(labelsize=8)
axes[1].text(0.05, 0.92, "(b)", transform=axes[1].transAxes, fontsize=12, fontweight="bold")

fig.tight_layout()
plt.show()

## No-scale vs full SUGRA Hessian

The no-scale potential has $\lambda = 0$ (Kähler moduli F-terms cancel $-3|W|^2$), while the full SUGRA potential has $\lambda = 3$. We compare the eigenvalue spectra.

In [ ]:
H_noscale = model.hessian(z_susy, zc_susy, tau_susy, tauc_susy, fl_susy, noscale=True)
H_full    = model.hessian(z_susy, zc_susy, tau_susy, tauc_susy, fl_susy, noscale=False)

eigs_ns = jnp.sort(jnp.linalg.eigvals(H_noscale).real)
eigs_full = jnp.sort(jnp.linalg.eigvals(H_full).real)

print(f"{'':>12} {'no-scale (λ=0)':>20} {'full SUGRA (λ=3)':>20}")
print("-" * 55)
for i in range(len(eigs_ns)):
    print(f"  eig_{i}:   {float(eigs_ns[i]):>+18.4e}   {float(eigs_full[i]):>+18.4e}")

print(f"\nDifference from -3|W|² correction: max|Δeig| = {float(jnp.max(jnp.abs(eigs_ns - eigs_full))):.4e}")

## Summary

| Hessian mode | Speed | Accuracy at generic pts | Use case |
|-------------|-------|------------------------|----------|
| `None` (autodiff) | Baseline | Exact (reference) | Default; always correct |
| `"SUSY"` | Fastest | Only at $D_AW=0$ | Large-scale sampling at SUSY vacua |
| `"SUGRA"` | ~Same as None | Machine precision | When you need explicit SUGRA building blocks |
| `"real"` | ~Same as None | Machine precision | When working in real coordinates |

- Use `model.mass_matrix(mode="SUSY")` for efficient mass computation at SUSY vacua
- Use `model.hessian(mode=None)` for guaranteed correctness at any point
- The mass hierarchy $m^2_{\mathrm{max}}/m^2_{\mathrm{min}}$ depends on the flux choice and geometry
- Tachyonic directions ($m^2 < 0$) signal instability — expected at non-SUSY critical points